## The Research Problem: Learning a 4-Quart Ising Spin Chain
Instead of a simple toy distribution, we will use a QCBM to learn the ground state probability distribution of a 1D Transverse-Field Ising Model (TFIM). This model is a cornerstone of condensed matter physics used to study quantum phase transitions.The Hamiltonian (energy equation) for this system is:$$H = -J \sum_{i=1}^{N-1} Z_i Z_{i+1} - h \sum_{i=1}^N X_i$$Where $J$ is the coupling energy between neighboring spins, and $h$ is an external transverse magnetic field. When $h \approx J$, the system undergoes a quantum phase transition, creating complex, highly entangled state distributions that are excellent benchmarks for generative AI.#

To bridge the gap between classroom exercises and active QML research, we can pivot from learning a simple binary pattern to Quantum State Tomography (QST) and Many-Body Physics Simulation.In quantum computing research, verifying that a physical QPU successfully created a complex quantum state is incredibly difficult because measuring a qubit destroys its superposition. To reconstruct the full quantum state classically, the number of measurements scales exponentially with the number of qubits ($2^N$).A major area of current research is using a QCBM as a Neural-Network Quantum State (NQS) equivalent—specifically, training a quantum circuit to learn and generalize the ground state wave function of a physical system (like a spin chain) from limited experimental data.

###Step 1: Generating the "Research" Target Data
In a real research scenario, this data would come from a physical experiment or an exact classical diagonalization. Here, we will use PennyLane's physics module to find the exact ground state of a 4-spin chain at the critical thermodynamic point ($J=1.0, h=1.0$) to act as our target distribution.

In [2]:
# Install required packages (quiet mode for cleaner output)
!pip install --quiet pennylane
!pip install --quiet amazon-braket-pennylane-plugin

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 76.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 388.0/388.0 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.8/245.8 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.5/144.5 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.3/1

In [3]:
import pennylane as qml
from pennylane import numpy as np

num_wires = 4

# Define the Transverse-Field Ising Model Hamiltonian
J = 1.0  # Spin interaction strength
h = 1.0  # Transverse field strength

coeffs = []
obs = []

# Internal coupling term (-J * Z_i * Z_{i+1})
for i in range(num_wires - 1):
    coeffs.append(-J)
    obs.append(qml.PauliZ(i) @ qml.PauliZ(i + 1))

# Transverse field term (-h * X_i)
for i in range(num_wires):
    coeffs.append(-h)
    obs.append(qml.PauliX(i))

Hamiltonian = qml.Hamiltonian(coeffs, obs)

# Compute the exact ground state probabilities classically for our target
matrix = qml.matrix(Hamiltonian)
eigenvalues, eigenvectors = np.linalg.eigh(matrix)
ground_state_vector = eigenvectors[:, 0]
target_distribution = np.abs(ground_state_vector) ** 2

print("Target Ground State Probabilities (16 states):")
print(np.round(target_distribution, 3))

Target Ground State Probabilities (16 states):
[0.23  0.065 0.028 0.05  0.028 0.012 0.021 0.065 0.065 0.021 0.012 0.028
 0.05  0.028 0.065 0.23 ]


###Step 2: The Strongly Entangling Research Circuit
A basic ansatz won't capture the complex correlations of a physical spin chain. In research, we use a Strongly Entangling Layers ansatz, which systematically circulates entangling gates across all pairs of qubits while applying arbitrary single-qubit rotations.

In [4]:
dev = qml.device("braket.local.qubit", wires=num_wires)

@qml.qnode(dev)
def quantum_generator(weights):
    # PennyLane offers pre-built research ansatzes
    # weights shape must match: (layers, num_wires, 3)
    qml.StronglyEntanglingLayers(weights=weights, wires=range(num_wires))
    return qml.probs(wires=range(num_wires))

###Step 3: Training with Total Variation Distance (TVD)
In physics research, Kullback-Leibler divergence can be unstable if the target state has true zero-probability states (which happens often in physics due to conservation laws). Instead, researchers frequently use Total Variation Distance (TVD) or Fidelity to evaluate generative accuracy.$$TVD(P, Q) = \frac{1}{2} \sum_{x} |P(x) - Q(x)|$$

In [ ]:
def total_variation_distance(probabilities, target):
    return 0.5 * np.sum(np.abs(probabilities - target))

def cost_function(weights):
    generated_probs = quantum_generator(weights)
    return total_variation_distance(generated_probs, target_distribution)

# Initialize a deep ansatz: 3 layers of strong entanglement
num_layers = 3
np.random.seed(101)
weights = np.random.uniform(0, 2 * np.pi, (num_layers, num_wires, 3), requires_grad=True)

# Using Adam optimizer, standard in research for multi-parameter landscapes
opt = qml.AdamOptimizer(stepsize=0.05)

print("Training QCBM on Ising Model Ground State...")
for step in range(201):
    weights, cost = opt.step_and_cost(cost_function, weights)
    if step % 20 == 0:
        print(f"Step {step:3d} | TVD Loss: {cost:.5f}")

Training QCBM on Ising Model Ground State...
Step   0 | TVD Loss: 0.44754
Step  20 | TVD Loss: 0.12824


###Step 4: Research Analysis & Verification
Once trained, researchers don't just look at a single metric. They look at whether the generative AI captured the macroscopic physical properties of the system, such as the Magnetization:$$\langle M_z \rangle = \frac{1}{N} \sum_{i=1}^N Z_i$$We can verify if our generative model accurately reproduces the physical observables of the true ground state.

In [ ]:
final_probs = quantum_generator(weights)

# Computational basis states mapped to their Z magnetization values
# e.g., '0000' -> +1, '1111' -> -1
def compute_magnetization(probs):
    mag = 0
    for i in range(2**num_wires):
        binary = format(i, f'0{num_wires}b')
        # Map '0' to +1 and '1' to -1
        spin_values = [1 if char == '0' else -1 for char in binary]
        mag += probs[i] * np.mean(spin_values)
    return mag

target_mag = compute_magnetization(target_distribution)
gen_mag = compute_magnetization(final_probs)

print("\n--- Research Verification ---")
print(f"Target System Magnetization:    {target_mag:.4f}")
print(f"Generated System Magnetization: {gen_mag:.4f}")
print(f"Final Model Accuracy (TVD):     {100 * (1 - cost):.2f}%")

**Hamiltonian-Driven Targets:** Instead of learning arbitrary human-defined datasets, the model is learning the natural probability landscape dictated by quantum mechanics.

**Scalability Testing:** Students can change num_wires = 4 to 8 or 12. They will quickly discover a core research bottleneck: gradient vanishing (Barren Plateaus), where the gradients of deep quantum circuits flatten out to zero, making training impossible without specialized initialization techniques.

**Hardware Constraints:** If you deploy this to a physical device on Amazon Braket (like Rigetti Ankaa-2), the presence of hardware noise will degrade the TVD score. A great undergraduate or graduate research paper topic involves comparing how different error-mitigation strategies affect a QCBM's ability to learn physical ground states under real-world hardware noise.